# Module 3: Data Cleaning

## Overview

This notebook implements the **Data Cleaning** module of the TrustGuard project.

The dataset is cleaned by handling missing values, removing duplicate records, standardizing text and date formats, performing data type conversions, and preparing a clean dataset. The processed data is stored as a Delta Table for analytical use.

In [0]:
# Import required PySpark functions

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Load transactions from Raw Layer

raw_df = spark.table("trustguard.raw_transactions")

# Display first 10 records

display(raw_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,true
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false


In [0]:
# Reject records where transaction_id or customer_id is missing

rejected_df = (
    raw_df
    .filter(
        col("transaction_id").isNull() |
        (trim(col("transaction_id")) == "") |
        col("customer_id").isNull() |
        (trim(col("customer_id")) == "")
    )
    .withColumn(
        "rejection_reason",
        lit("Missing transaction_id or customer_id")
    )
)

print("Rejected Records:", rejected_df.count())

display(rejected_df.limit(10))

Rejected Records: 0


transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,rejection_reason


In [0]:
# Keep records having valid transaction_id and customer_id

clean_df = (
    raw_df
    .filter(
        col("transaction_id").isNotNull() &
        (trim(col("transaction_id")) != "") &
        col("customer_id").isNotNull() &
        (trim(col("customer_id")) != "")
    )
)

print("Valid Records:", clean_df.count())

Valid Records: 12575


In [0]:
# Calculate average values for missing numeric fields

average_values = clean_df.select(
    avg("price_per_unit").alias("avg_price"),
    avg("quantity").alias("avg_quantity")
).first()

avg_price = float(average_values["avg_price"])
avg_quantity = int(average_values["avg_quantity"])

# Fill missing text and numeric values

clean_df = clean_df.fillna({
    "category": "Unknown",
    "item": "Unknown",
    "payment_method": "Unknown",
    "location": "Unknown",
    "discount_applied": "No",
    "price_per_unit": avg_price,
    "quantity": avg_quantity
})

In [0]:
# Convert numeric columns into correct data types

clean_df = (
    clean_df
    .withColumn("price_per_unit", col("price_per_unit").cast("double"))
    .withColumn("quantity", col("quantity").cast("integer"))
    .withColumn("total_spent", col("total_spent").cast("double"))
)

clean_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- category: string (nullable = false)
 |-- item: string (nullable = false)
 |-- price_per_unit: double (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- location: string (nullable = false)
 |-- transaction_date: date (nullable = true)
 |-- discount_applied: boolean (nullable = true)



In [0]:
# Standardize date and text columns

clean_df = (
    clean_df
    .withColumn(
        "transaction_date",
        coalesce(
            to_date(col("transaction_date"), "yyyy-MM-dd"),
            to_date(col("transaction_date"), "MM/dd/yyyy"),
            to_date(col("transaction_date"), "M/d/yyyy"),
            to_date(col("transaction_date"), "dd-MM-yyyy"),
            to_date(col("transaction_date"), "dd/MM/yyyy")
        )
    )
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("item", initcap(trim(col("item"))))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("location", initcap(trim(col("location"))))
    .withColumn("discount_applied", initcap(trim(col("discount_applied"))))
)

display(clean_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_pat,18.5,10,185.0,Digital Wallet,Online,2024-04-08,True
TXN_3731986,CUST_22,Milk Products,Item_17_milk,29.0,9,261.0,Digital Wallet,Online,2023-07-23,True
TXN_9303719,CUST_02,Butchers,Item_12_but,21.5,2,43.0,Credit Card,Online,2022-10-05,False
TXN_9458126,CUST_06,Beverages,Item_16_bev,27.5,9,247.5,Credit Card,Online,2022-05-07,False
TXN_4575373,CUST_05,Food,Item_6_food,12.5,7,87.5,Digital Wallet,Online,2022-10-02,False
TXN_7482416,CUST_09,Patisserie,Unknown,23.365911749958215,10,200.0,Credit Card,Online,2023-11-30,False
TXN_3652209,CUST_07,Food,Item_1_food,5.0,8,40.0,Credit Card,In-store,2023-06-10,True
TXN_1372952,CUST_21,Furniture,Unknown,33.5,5,null,Digital Wallet,In-store,2024-04-02,True
TXN_9728486,CUST_23,Furniture,Item_16_fur,27.5,1,27.5,Credit Card,In-store,2023-04-26,False
TXN_2722661,CUST_25,Butchers,Item_22_but,36.5,3,109.5,Cash,Online,2024-03-14,False


In [0]:
# Fill missing total_spent using price_per_unit × quantity

clean_df = clean_df.withColumn(
    "total_spent",
    when(
        col("total_spent").isNull(),
        col("price_per_unit") * col("quantity")
    ).otherwise(col("total_spent"))
)

display(clean_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_pat,18.5,10,185.0,Digital Wallet,Online,2024-04-08,True
TXN_3731986,CUST_22,Milk Products,Item_17_milk,29.0,9,261.0,Digital Wallet,Online,2023-07-23,True
TXN_9303719,CUST_02,Butchers,Item_12_but,21.5,2,43.0,Credit Card,Online,2022-10-05,False
TXN_9458126,CUST_06,Beverages,Item_16_bev,27.5,9,247.5,Credit Card,Online,2022-05-07,False
TXN_4575373,CUST_05,Food,Item_6_food,12.5,7,87.5,Digital Wallet,Online,2022-10-02,False
TXN_7482416,CUST_09,Patisserie,Unknown,23.365911749958215,10,200.0,Credit Card,Online,2023-11-30,False
TXN_3652209,CUST_07,Food,Item_1_food,5.0,8,40.0,Credit Card,In-store,2023-06-10,True
TXN_1372952,CUST_21,Furniture,Unknown,33.5,5,167.5,Digital Wallet,In-store,2024-04-02,True
TXN_9728486,CUST_23,Furniture,Item_16_fur,27.5,1,27.5,Credit Card,In-store,2023-04-26,False
TXN_2722661,CUST_25,Butchers,Item_22_but,36.5,3,109.5,Cash,Online,2024-03-14,False


In [0]:
# Check remaining NULL values

clean_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in clean_df.columns
]).show()

+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+
|transaction_id|customer_id|category|item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+
|             0|          0|       0|   0|             0|       0|          0|             0|       0|               0|               0|
+--------------+-----------+--------+----+--------------+--------+-----------+--------------+--------+----------------+----------------+



In [0]:
# Remove duplicate transaction IDs

before_count = clean_df.count()

clean_df = clean_df.dropDuplicates(["transaction_id"])

after_count = clean_df.count()

print("Rows Before Deduplication:", before_count)
print("Rows After Deduplication:", after_count)
print("Duplicates Removed:", before_count - after_count)

Rows Before Deduplication: 12575
Rows After Deduplication: 12575
Duplicates Removed: 0


In [0]:
# Flag rows where total_spent != quantity × price_per_unit

clean_df = clean_df.withColumn(
    "amount_validation",
    when(
        abs(col("total_spent") - (col("price_per_unit") * col("quantity"))) > 0.01,
        "Mismatch"
    ).otherwise("Match")
)

display(clean_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,amount_validation
TXN_6867343,CUST_09,Patisserie,Item_10_pat,18.5,10,185.0,Digital Wallet,Online,2024-04-08,True,Match
TXN_3731986,CUST_22,Milk Products,Item_17_milk,29.0,9,261.0,Digital Wallet,Online,2023-07-23,True,Match
TXN_9303719,CUST_02,Butchers,Item_12_but,21.5,2,43.0,Credit Card,Online,2022-10-05,False,Match
TXN_9458126,CUST_06,Beverages,Item_16_bev,27.5,9,247.5,Credit Card,Online,2022-05-07,False,Match
TXN_4575373,CUST_05,Food,Item_6_food,12.5,7,87.5,Digital Wallet,Online,2022-10-02,False,Match
TXN_7482416,CUST_09,Patisserie,Unknown,23.365911749958215,10,200.0,Credit Card,Online,2023-11-30,False,Mismatch
TXN_3652209,CUST_07,Food,Item_1_food,5.0,8,40.0,Credit Card,In-store,2023-06-10,True,Match
TXN_1372952,CUST_21,Furniture,Unknown,33.5,5,167.5,Digital Wallet,In-store,2024-04-02,True,Match
TXN_9728486,CUST_23,Furniture,Item_16_fur,27.5,1,27.5,Credit Card,In-store,2023-04-26,False,Match
TXN_2722661,CUST_25,Butchers,Item_22_but,36.5,3,109.5,Cash,Online,2024-03-14,False,Match


In [0]:
# Create Customer Dimension

customers_df = (
    clean_df
    .groupBy("customer_id")
    .agg(
        first("location").alias("location"),
        count("transaction_id").alias("total_orders")
    )
)

print("Total Customers:", customers_df.count())

display(customers_df.limit(10))

Total Customers: 25


customer_id,location,total_orders
CUST_09,Online,519
CUST_22,Online,501
CUST_02,Online,488
CUST_06,Online,481
CUST_05,Online,544
CUST_07,In-store,491
CUST_21,In-store,498
CUST_23,In-store,513
CUST_25,Online,476
CUST_14,In-store,484


In [0]:
# Check referential integrity

referential_failure_df = (
    clean_df.alias("t")
    .join(
        customers_df.alias("c"),
        on="customer_id",
        how="left_anti"
    )
)

print("Referential Integrity Failures:", referential_failure_df.count())

Referential Integrity Failures: 0


In [0]:
# Save Clean Transactions

clean_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("trustguard.clean_transactions")

# Save Customer Dimension

customers_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("trustguard.clean_customers")

print("Tables Saved Successfully.")

Tables Saved Successfully.


In [0]:
display(spark.table("trustguard.clean_transactions").limit(10))

display(spark.table("trustguard.clean_customers").limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,amount_validation
TXN_8425168,CUST_25,Milk Products,Item_16_milk,27.5,7,192.5,Cash,In-store,2022-03-12,False,Match
TXN_5885894,CUST_01,Patisserie,Item_14_pat,24.5,10,245.0,Credit Card,In-store,2022-02-09,False,Match
TXN_3671271,CUST_18,Patisserie,Item_12_pat,21.5,8,172.0,Cash,In-store,2023-11-27,False,Match
TXN_9908330,CUST_19,Food,Item_18_food,30.5,4,122.0,Credit Card,Online,2022-09-29,True,Match
TXN_2780662,CUST_02,Milk Products,Item_19_milk,32.0,10,320.0,Digital Wallet,In-store,2023-01-19,False,Match
TXN_2646301,CUST_18,Furniture,Item_24_fur,39.5,9,355.5,Credit Card,In-store,2022-12-07,False,Match
TXN_5328604,CUST_07,Food,Item_20_food,33.5,10,335.0,Cash,Online,2024-07-08,False,Match
TXN_8219228,CUST_17,Computers And Electric Accessories,Item_10_cea,18.5,5,92.5,Cash,Online,2022-05-27,False,Match
TXN_1112365,CUST_19,Food,Item_24_food,39.5,10,395.0,Cash,Online,2023-09-24,False,Match
TXN_2953434,CUST_25,Furniture,Item_25_fur,41.0,10,410.0,Credit Card,In-store,2023-08-10,False,Match


customer_id,location,total_orders
CUST_17,Online,487
CUST_12,Online,498
CUST_22,Online,501
CUST_13,Online,534
CUST_04,Online,474
CUST_10,In-store,501
CUST_25,In-store,476
CUST_23,Online,513
CUST_01,In-store,507
CUST_16,In-store,515


In [0]:
spark.table("trustguard.clean_transactions") \
.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", "true") \
.csv("/Volumes/workspace/default/trustguard_data/clean_transactions")

In [0]:
spark.table("trustguard.clean_customers") \
.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", "true") \
.csv("/Volumes/workspace/default/trustguard_data/clean_customers")

In [0]:
clean_df.groupBy("amount_validation").count().show()

+-----------------+-----+
|amount_validation|count|
+-----------------+-----+
|            Match|11966|
|         Mismatch|  609|
+-----------------+-----+

